# Sleep State Detection - Refactored

This notebook demonstrates the refactored, modular approach to sleep state detection.

## 1. Setup and Imports

In [ ]:
# Add src to path to enable imports
import sys
sys.path.insert(0, '.')

import pandas as pd
from src import data_loader, feature_engineering, model, utils
from src.config import EVENT_LABEL_MAP

## 2. Configure Data Paths

Set the paths to your data files. Update these paths based on your environment.

In [ ]:
# Configure your data paths here
TRAIN_SERIES_PATH = "/path/to/train_series.parquet"  # Update this path
TRAIN_EVENTS_PATH = "/path/to/train_events.csv"      # Update this path
TEST_SERIES_PATH = "/path/to/test_series.parquet"    # Update this path
SUBMISSION_PATH = "submission.csv"

## 3. Load and Explore Data

In [ ]:
# Load training data
print("Loading training data...")
train_series, train_events = data_loader.load_train_data(
    TRAIN_SERIES_PATH,
    TRAIN_EVENTS_PATH,
    clean_na=True
)

# Display data summaries
data_loader.print_data_summary(train_series, "Training Series")
data_loader.print_data_summary(train_events, "Training Events")

# Display first few rows
print("\nFirst rows of training series:")
print(train_series.head())
print("\nFirst rows of training events:")
print(train_events.head())

## 4. Merge Data

In [ ]:
# Merge series and events data
print("Merging training data...")
merged_data = data_loader.merge_series_and_events(train_series, train_events)
data_loader.print_data_summary(merged_data, "Merged Data")

print("\nFirst rows of merged data:")
print(merged_data.head())

# Free memory
utils.clear_memory(train_series)

## 5. Feature Engineering

In [ ]:
# Apply all feature engineering steps
print("Engineering features...")
merged_data = feature_engineering.prepare_features(
    merged_data,
    is_training=True,
    sensor_columns=['anglez', 'enmo']
)

print("\nFeatures after engineering:")
print(merged_data.head(10))
print("\nColumns:", list(merged_data.columns))

## 6. Train and Evaluate Model

In [ ]:
# Train model using the complete pipeline
print("Training model...")
trained_model, eval_results = model.train_and_evaluate_pipeline(
    merged_data,
    verbose=True
)

print(f"\nFinal validation accuracy: {eval_results['accuracy']:.4f}")

## 7. Load and Process Test Data

In [ ]:
# Load test data
print("Loading test data...")
test_series = data_loader.load_test_data(TEST_SERIES_PATH)
data_loader.print_data_summary(test_series, "Test Series")

print("\nFirst rows of test data:")
print(test_series.head())

In [ ]:
# Apply feature engineering to test data
print("Engineering test features...")
test_series = feature_engineering.prepare_features(
    test_series,
    is_training=False,
    sensor_columns=['anglez', 'enmo']
)

# Drop original sensor columns to save memory (optional)
test_series = utils.drop_columns_inplace(test_series, ['anglez', 'enmo'])

print("\nTest features after engineering:")
print(test_series.head(10))

## 8. Make Predictions

In [ ]:
# Prepare features for prediction
test_features = model.prepare_features_for_model(test_series)

# Make predictions with confidence scores
print("Making predictions...")
predicted_labels, confidence_scores = model.predict_with_confidence(
    trained_model,
    test_features
)

# Add predictions to DataFrame
test_series = utils.add_predictions_to_dataframe(
    test_series,
    predicted_labels,
    confidence_scores,
    label_map=EVENT_LABEL_MAP
)

print("\nPredictions:")
print(test_series[['hour', 'predicted_event_label', 'event', 'confidence_score']].head(20))

## 9. Export Results

In [ ]:
# Export submission file (customize columns as needed)
# utils.export_submission(test_series, SUBMISSION_PATH)

print("\nWorkflow complete!")
print(f"Memory usage: {utils.get_memory_usage(test_series):.2f} MB")